# 8. 循环神经网络 (Recurrent Neural Networks, RNN)

> 如果说卷积神经网络（CNN）是深度学习的“双眼”，擅长捕捉**空间**结构；
> 
> 那么循环神经网络（RNN）就是深度学习的“大脑”，擅长处理**时间**序列。

1. **核心矛盾：从“独立”到“关联”**

    - **之前的模型（MLP/CNN）**：假设样本之间是**独立同分布 (IID)** 的。输入一张猫，输出猫；输入下一张狗，与上一张无关。
    - **现实的序列**：在文本、语音、股市中，当前的数值极度依赖于过去。

      - 例子：“我吃了一个____”。（根据前文，这里大概率是“苹果”或“馒头”，而不是“飞机”）。

    - **RNN 的使命**：引入**隐藏状态 (Hidden State)**，让模型具备“记忆”能力。

2. **知识版图：本章的四个核心模块**
  
    - 第一模块：文本预处理 (Text Preprocessing)

        在把文字喂给神经网络前，必须经过洗练：
        1. **词元化 (Tokenization)**：把句子拆成单词或字符。
        2. **词表 (Vocabulary)**：建立“单词”到“数字 ID”的映射。
        3. **加载序列数据**：学习如何通过“随机采样”或“顺序分区”来切分长文本。

     - 第二模块：语言模型 (Language Models)
       
       1. **数学本质**：计算一个序列出现的概率 $P(x_1, x_2, \dots, x_T)$。
       2. **马尔可夫模型 (Markov Models)**：理解 $n$ 元语法 (n-grams) 的局限。
       3. **齐普夫定律 (Zipf's Law)**：理解真实语言中词频分布的极端不平衡（二八定律）。

     - 第三模块：循环神经网络架构 (RNN Architecture)
       
       1. **隐藏状态 ($H_t$)**：理解 $H_t = \phi(X_t W_{xh} + H_{t-1} W_{hh} + b_h)$。
       2. **循环特性**：理解为什么 $W_{hh}$（隐藏层到隐藏层的权重）是实现记忆的关键。
       3. **困惑度 (Perplexity)**：学习评价序列模型好坏的标准（不再仅仅是准确率）。

     - 第四模块：随时间反向传播 (BPTT)
       
       1. **核心痛点**：序列越长，计算图越深。
       2. **梯度危机**：深度理解为什么 RNN 天生容易产生**梯度消失**或**梯度爆炸**。
       3. **截断梯度**：学习工程上如何通过“梯度裁剪”来保证训练不崩溃。

---


## 8.1 序列模型 (Sequence Models)

### 1. 核心数学矛盾：维度的无穷增长

在之前的模型（如 MLP/CNN）中，我们处理的是固定维度的输入。但在序列数据中，为了预测 $t$ 时刻的输出 $x_t$，理论上我们需要考虑之前所有的观测值 $x_1, \dots, x_{t-1}$。

#### 1.1 概率链式法则 (Chain Rule)
任意序列的联合概率分布可以展开为：
$$P(x_1, \dots, x_T) = \prod_{t=1}^T P(x_t \mid x_1, \dots, x_{t-1})$$
- **工程挑战**：随着时间 $t$ 的推移，条件概率的输入向量 $x_1, \dots, x_{t-1}$ 的长度会不断增加，模型无法处理变长的、且趋于无穷大的特征向量。

---

### 2. 解决方案：简化假设

为了使计算可行，序列模型引入了两种核心策略：

#### 2.1 自回归模型 (Autoregressive Models)

假设当前值只取决于过去固定跨度 $\tau$ 内的观测值。
- **数学表达**：$P(x_t \mid x_{t-1}, \dots, x_1) \approx P(x_t \mid x_{t-1}, \dots, x_{t-\tau})$
- **术语**：这被称为 **$\tau$ 阶马尔可夫模型 (Markov Model)**。此时，特征的维度被固定为 $\tau$。

#### 2.2 潜变量模型 (Latent Variable Models)
不保留长长的历史，而是维护一个**隐藏状态 (Hidden State)** $h_t$ 来总结历史。
- **公式**：
    1.  $h_t = g(h_{t-1}, x_{t-1})$ （更新记忆）
    2.  $\hat{x}_t = P(x_t \mid h_t)$ （基于记忆预测）
- **意义**：这是 RNN 的理论基石。

---

### 3. 核心约束：因果性 (Causality)

**因果性**是序列建模的铁律：**预测 $x_t$ 时，模型只能接触到时间轴上 $t$ 以前的数据。**
- **体现**：在构造数据集时，标签必须是特征窗口之后的第一个点。
- **后果**：模型严禁“偷看未来”，任何违反此规则的设计都会导致训练出的模型在实际应用中毫无用处（数据泄露）。

---

### 4. 构造序列数据集

我们将通过预测带噪声的正弦波来演示如何构造符合因果性的训练集。


In [1]:
import torch
from torch import Tensor

class SequenceDataGenerator:
    """序列数据生成与处理器。
    
    负责生成模拟序列数据，并利用滑动窗口技术构造符合因果性约束的特征-标签对。
    """
    def __init__(self, n_points: int = 1000, tau: int = 4):
        """
        Args:
            n_points: 总数据点数。
            tau: 时间窗口大小（马尔可夫阶数）。
        """
        self.n_points = n_points
        self.tau = tau
        self.time = torch.arange(1, n_points + 1, dtype=torch.float32)
        # 生成带噪声的正弦波
        self.x = torch.sin(0.01 * self.time) + torch.normal(0, 0.2, (n_points,))

    def split_data(self, train_percent: float = 0.6) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        """构造滑动窗口数据集并划分训练集与验证集。
        
        因果性体现：X[i] = [x[i], ..., x[i+tau-1]], y[i] = x[i+tau]。
        
        Returns:
            train_features: 形状为 (n_points - tau, tau) 的训练集特征矩阵。
            train_labels: 形状为 (n_points - tau, 1) 的训练集标签向量。
            val_features: 验证集特征矩阵。
            val_labels: 验证集标签向量。
        """
        num_samples = self.n_points - self.tau
        features = torch.zeros((num_samples, self.tau))
        
        # 填充特征矩阵：每一列都是序列的一个偏移版本
        for i in range(self.tau):
            features[:, i] = self.x[i : i + num_samples]
            
        # 标签是紧随窗口后的那个点，即从索引 tau 开始的所有点
        labels = self.x[self.tau:].reshape((-1, 1))
        
        # 按照 train_percent 划分数据集
        n_train = int(num_samples * train_percent)
        train_features = features[:n_train]
        train_labels = labels[:n_train]
        val_features = features[n_train:]
        val_labels = labels[n_train:]
        
        return train_features, train_labels, val_features, val_labels

In [2]:
# --- 验证数据构造 ---
gen = SequenceDataGenerator(tau=4)
train_features, train_labels, _, _ = gen.split_data()
print(f"特征示例 (前4个点): {train_features[0]}")
print(f"标签示例 (第5个点): {train_labels[0]}")

特征示例 (前4个点): tensor([-0.2403,  0.0826, -0.1106,  0.2739])
标签示例 (第5个点): tensor([-0.1497])



---

### 5. 多步预测 (Multistep Prediction) 的崩溃

这是 8.1 节最重要的结论，揭示了简单自回归模型的局限性。

#### 5.1 单步预测 (One-step-ahead)

-  **定义**：始终使用**真实**的历史数据预测下一个点。
-  **表现**：由于输入是准确的，模型表现通常非常优秀。

#### 5.2 多步预测 (Multistep / K-step-ahead)

-  **定义**：使用模型**自己生成的预测值**作为输入，继续预测更远的未来。
-  **表现**：**误差累积 (Error Accumulation)**。
-  **数学直觉**：如果单步预测有误差 $\epsilon$，在 $k$ 步预测中，误差会按指数级传播。你会发现预测曲线迅速偏离真实轨迹。

---

### 6. 总结 Checklist

-  **自回归策略**：理解如何将变长序列转化为固定长度的特征向量。
-  **因果性约束**：掌握滑动窗口切片时“不准看未来”的索引逻辑。
-  **马尔可夫阶数 ($\tau$)**：明白窗口大小决定了模型“回看”历史的深度。
-  **误差累积**：深刻理解为什么简单的自回归模型无法进行长程预测（多步预测会崩溃）。

---


## 8.2 文本预处理 (Text Preprocessing)

> 在 NLP（自然语言处理）中，我们常面对的是非结构化的文本。
> 
> 神经网络无法直接处理“单词”，因此我们需要一套标准化的流程，将原始文本转化为模型可计算的数字索引。
>
> 本节我们将实现一个完整的**文本预处理流水线**，涵盖读取、词元化、词表构建和索引转换。

### 1. 核心工作流 (The Pipeline)

一个标准的文本预处理流程包含四个步骤：
1. **读取文本**：将原始文件加载到内存。
2. **词元化 (Tokenization)**：将字符串拆分为词元（Tokens）。
3. **构建词表 (Vocabulary)**：建立词元到数字 ID 的映射。
4. **语料转换 (Corpus Mapping)**：将文本序列转换为数字索引序列。

---

### 2. 构建预处理组件

我们将使用 H.G. Wells 的《时光机器》（The Time Machine）作为实验语料。

#### 2.1 读取与初步清洗


In [3]:
import re
import requests
import hashlib
from pathlib import Path
from typing import Union

def download_time_machine(data_dir: Union[str, Path] = 'temp/data/plaintext') -> Path:
    """幂等下载《时光机器》数据集。
    
    Args:
        data_dir: 数据存储目录。
        
    Returns:
        Path: 下载后的文件 Path 对象。
        
    Raises:
        RuntimeError: 下载失败或网络异常时抛出。
    """
    url = 'http://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt'
    expected_sha1 = '090b5e7e70c295757f55df93cb0a180b9691891a'
    
    data_path = Path(data_dir)
    data_path.mkdir(parents=True, exist_ok=True)
    file_path = data_path / 'timemachine.txt'
    
    # 幂等校验逻辑
    if file_path.exists():
        content = file_path.read_bytes()
        if hashlib.sha1(content).hexdigest() == expected_sha1:
            print(f"Dataset already exists and verified: {file_path}")
            return file_path
    
    # 执行下载
    print(f"Downloading dataset from {url}...")
    try:
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()
        
        # 写入文件并验证
        file_path.write_bytes(response.content)
        
        # 下载后的二次校验
        if hashlib.sha1(file_path.read_bytes()).hexdigest() != expected_sha1:
            file_path.unlink(missing_ok=True)
            raise RuntimeError("Data integrity check failed after download.")
            
        print(f"Download complete: {file_path}")
        
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Failed to download dataset: {e}")
        
    return file_path

def read_time_machine() -> list[str]:
    """读取《时光机器》数据集并逐行进行标准化清洗。

    Returns:
        list[str]: 包含清洗后文本行的列表。
    """
    file_path = download_time_machine()
    
    # 使用 utf-8 编码读取并处理
    with file_path.open('r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # 清洗逻辑：正则过滤、去除冗余空格、统一转小写
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

In [4]:
lines = read_time_machine()
print(f'# 文本总行数: {len(lines)}')
print(lines[0])
print(lines[10])

Dataset already exists and verified: temp/data/plaintext/timemachine.txt
# 文本总行数: 3221
the time machine by h g wells
twinkled and his usually pale face was flushed and animated the


#### 2.2 词元化 (Tokenization)

词元是文本处理的最小单位。


In [5]:
def tokenize(lines: list[str], token_type: str = 'word') -> list[list[str]]:
    """将文本行拆分为词元。

    Args:
        lines: 文本行列表。
        token_type: 词元类型，'word'（单词）或 'char'（字符）。

    Returns:
        list[list[str]]: 每一行对应的词元列表。
    """
    if token_type == 'word':
        return [line.split() for line in lines]
    elif token_type == 'char':
        return [list(line) for line in lines]
    else:
        raise ValueError(f"错误：不支持的词元类型 {token_type}")
    
tokens = tokenize(lines,)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


#### 2.3 词表类 (The Vocab Class)

这是本节最核心的工程组件。它负责维护 `Token <-> Index` 的映射。


In [6]:
import collections
from typing import Any, Optional, Counter as CounterType

class Vocab:
    """文本词表类。

    负责统计词频、过滤低频词，并提供词元与索引的双向映射。

    Attributes:
        token_to_idx (dict[str, int]): 词元到索引的映射。
        idx_to_token (list[str]): 索引到词元的映射。
    """
    def __init__(
        self, 
        tokens: Optional[list[Union[str, list[str]]]] = None, 
        min_freq: int = 0, 
        reserved_tokens: Optional[list[str]] = None
    ) -> None:
        """初始化词表。

        Args:
            tokens: 词元列表或嵌套列表。
            min_freq: 最小词频阈值，低于此值的词元将被标记为 <unk>。
            reserved_tokens: 预留的特殊词元（如 <pad>, <bos>, <eos>）。
        """
        if tokens is None: tokens = []
        if reserved_tokens is None: reserved_tokens = []

        # 1. 统计词频 (并降序排序)
        flatten_tokens = self._flatten(tokens)
        counter: CounterType[str] = collections.Counter(flatten_tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.freq_dict = dict(counter)

        # 2. 初始映射（<unk> 的索引永远是 0）
        self.idx_to_token: list[str] = ['<unk>'] + reserved_tokens
        self.token_to_idx: dict[str, int] = {token: i for i, token in enumerate(self.idx_to_token)}

        # 3. 构建索引
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self) -> int:
        """返回词表大小。"""
        return len(self.idx_to_token)

    def __getitem__(self, tokens: Union[str, list[str], tuple[str, ...]]) -> Union[int, list[int]]:
        """根据词元获取索引。"""
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.token_to_idx['<unk>'])
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices: Union[int, list[int]]) -> Union[str, list[str]]:
        """根据索引获取词元。"""
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    def _flatten(self, tokens: Any) -> list[str]:
        """递归展平嵌套的词元列表。"""
        result = []
        for item in tokens:
            if isinstance(item, list):
                result.extend(self._flatten(item))
            else:
                result.append(item)
        return result

    @property
    def unk(self) -> int:
        """返回未知词元的索引。"""
        return 0
    
    @property
    def token_freqs(self):
        """返回词频。"""
        return self._token_freqs
    
    def frequency(self, token: str) -> int:
        """查询特定词的出现次数。"""
        return self.freq_dict.get(token, 0)

In [7]:
from rich import print
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])

[
    ('<unk>', 0),
    ('the', 1),
    ('i', 2),
    ('and', 3),
    ('of', 4),
    ('a', 5),
    ('to', 6),
    ('was', 7),
    ('in', 8),
    ('that', 9)
]


---

### 3. 整合：语料库加载器


In [8]:
def load_corpus_time_machine(
    max_tokens: int = -1, 
    token_type: str = 'char'
) -> tuple[list[int], Vocab]:
    """加载《时光机器》语料库并转换为索引。

    Args:
        max_tokens: 截断的最大长度 (若非正，则全部)。
        token_type: 'char' 或 'word'。

    Returns:
        tuple[list[int], Vocab]: 一个包含以下内容的元组：
            - corpus: 索引化的长序列。
            - vocab: 构建好的词表。
    """
    lines = read_time_machine()
    tokens = tokenize(lines, token_type)
    vocab = Vocab(tokens)
    
    # 展平所有行，形成一个单一的长序列
    corpus = [vocab[token] for line in tokens for token in line]
    
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

# --- 验证程序 ---
corpus, vocab = load_corpus_time_machine(token_type='word')
print(f"词表大小: {len(vocab)}")
print(f"语料库前10个索引: {corpus[:10]}")
print(f"还原后的文本: {' '.join(vocab.to_tokens(corpus[:10]))}")

Dataset already exists and verified: temp/data/plaintext/timemachine.txt

词表大小: 4580

语料库前10个索引: [1, 19, 50, 40, 2183, 2184, 400, 2, 1, 19]

还原后的文本: the time machine by h g wells i the time


---

### 4. 细致梳理

1. **词元化策略对比**：
    - **单词级 (Word)**：语义明确，但词表很大（通常几万个），且容易遇到未知词（OOV）。
    - **字符级 (Char)**：词表极小（几十到几百个），没有 OOV 问题，但单个词元语义缺失，序列长度大幅增加。
2.  **特殊词元 (Special Tokens)**：
    - `<unk>`：Unknown，代表词表中未出现的词。
    - `<pad>`：Padding，用于填充句子到固定长度。
    - `<bos>` / `<eos>`：句子的开始和结束。
3.  **词频过滤 (Min Frequency)**：
    - **工程意义**：训练数据中只出现一两次的词（长尾词）通常是噪声或拼写错误，过滤它们能显著减小模型参数量并提高泛化能力。

---

### 5. 关键工程暗知识 (Engineering Wisdom)

-  **`Counter` 的效率**：在处理百万级语料时，`collections.Counter` 是 Python 中统计词频的最快方式。
-  **内存优化**：在 `Vocab` 中使用 `token_to_idx`（字典）和 `idx_to_token`（列表）是空间与时间效率的最优组合，支持 $O(1)$ 级别的双向查找。
-  **正则化清洗**：在 8.2 节中，我们只保留字母。在实际工程（如处理中文）中，还需要考虑分词（jieba）、去除停用词等步骤。

---

### 6. 总结 Checklist

-  **Tokenization**：理解词与字符的区别。
-  **Vocab 映射**：掌握如何将文本数字化。
-  **特殊词元**：明白 `<unk>` 的必要性。

---


## 8.3 语言模型和数据集 (Language Models and Dataset)

### 1. 核心任务：预测下一个词元

在 8.2 中，我们已经把文本变成了离散词元序列。接下来要解决的问题是：**给定前文，如何预测下一个词元？**

如果把词元序列记作 $x_1, x_2, \dots, x_T$，那么语言模型的目标就是计算序列 $x_1, x_2, \dots, x_T$ 的联合概率：

$$
P(x_1, x_2, \dots, x_T)
$$

根据概率链式法则，它可以展开为：

$$
P(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, x_2, \dots, x_{t-1})
$$

这说明：**只要我们能建模“下一个词元在前文条件下出现的概率”，就等价于建好了整个序列的概率模型。**

#### 1.1. $n$ 元语法 (n-gram)
由于历史信息过长会导致计算量爆炸，我们引入马尔可夫假设：

- **一元语法 (Unigram)**：$P(x_t \mid x_{t-1}, \dots) \approx P(x_t)$（词之间相互独立）。
- **二元语法 (Bigram)**：$P(x_t \mid x_{t-1}, \dots) \approx P(x_t \mid x_{t-1})$（只取决于前一个词）。
- **三元语法 (Trigram)**：$P(x_t \mid x_{t-1}, \dots) \approx P(x_t \mid x_{t-1}, x_{t-2})$。

#### 1.2. $n-gram$ 的实现


In [9]:
import collections

def count_corpus(tokens: list[str] | list[list[str]]) -> collections.Counter[str]:
    """统计语料中的词元频率。

    Args:
        tokens: 一维或二维词元列表。

    Returns:
        词元到出现次数的 Counter 映射。
    """
    if len(tokens) == 0 or isinstance(tokens[0], str):
        flat_tokens = tokens
    else:
        flat_tokens = [token for line in tokens for token in line]

    return collections.Counter(flat_tokens)

# 使用字符级语料，更贴近 D2L 后续 RNN 训练设置
char_corpus, char_vocab = load_corpus_time_machine(max_tokens=10000, token_type='char')
char_tokens = [char_vocab.to_tokens(index) for index in char_corpus]

unigram_freqs = count_corpus(char_tokens)
bigram_tokens = [
    ''.join(pair)
    for pair in zip(char_tokens[:-1], char_tokens[1:])
]
trigram_tokens = [
    ''.join(triple)
    for triple in zip(char_tokens[:-2], char_tokens[1:-1], char_tokens[2:])
]

bigram_freqs = count_corpus(bigram_tokens)
trigram_freqs = count_corpus(trigram_tokens)

print('字符级词表大小:', len(char_vocab))
print('Top-10 unigram:', unigram_freqs.most_common(10))
print('Top-10 bigram:', bigram_freqs.most_common(10))
print('Top-10 trigram:', trigram_freqs.most_common(10))


Dataset already exists and verified: temp/data/plaintext/timemachine.txt

字符级词表大小: 28

Top-10 unigram:
[
    (' ', 1684),
    ('e', 1017),
    ('t', 802),
    ('a', 708),
    ('i', 629),
    ('o', 613),
    ('n', 593),
    ('s', 514),
    ('h', 460),
    ('r', 455)
]

Top-10 bigram:
[
    ('e ', 353),
    (' t', 296),
    ('th', 266),
    (' a', 230),
    ('t ', 200),
    ('s ', 190),
    ('he', 181),
    ('d ', 150),
    ('an', 148),
    ('in', 133)
]

Top-10 trigram:
[
    (' th', 190),
    ('the', 152),
    ('he ', 108),
    (' an', 73),
    ('e t', 69),
    ('ime', 60),
    ('is ', 58),
    ('and', 57),
    (' of', 55),
    ('nd ', 54)
]

从上面的频数可以看出：

1. **Unigram 只能看出“高频字符”**，比如空格、常见字母。
2. **Bigram 开始捕获局部搭配**，比如某些字符对会频繁一起出现。
3. **Trigram 语义更强，但稀疏性更严重**：组合数暴涨，很多三元组只出现一次。

这也是为什么仅靠 N-gram 很快会撞到天花板：上下文窗口一大，统计量就会指数爆炸。RNN 的目标，就是用参数共享的方式去近似这类条件概率。

---

### 2. 齐普夫定律 (Zipf's Law)

在真实语言中，词频的分布极不均匀。
- **定律内容**：第 $i$ 个最常用词的频率 $f_i$ 与其排名 $i$ 成反比：$f_i \propto i^{-\alpha}$ (通常 $\alpha \approx 1$ )。
- **工程启示**：
    1.  极少数词（如 "the", "a"）占据了语料库的大部分。
    2.  大部分词（长尾词）出现频率极低。
    3.  **结论**：即便使用二元或三元语法，大多数高阶组合在语料库中也从未出现过，因此我们需要将低频词 (如频率 < 5) 统一替换为 \<unk\> ，或者采用深度学习模型（如 RNN）来处理这种稀疏性。

---

### 3. 序列数据的采样

在训练 RNN 时，我们不能简单地随机打乱所有字符也不能将所有字符一次喂进去。我们需要将长序列切分成等长的**子序列（Subsequences）**。

以下是两种主流采样方法。

#### 3.1 随机采样 (Random Sampling)

把语料切成许多互不连续的小片段，再随机打乱组成 batch。

- **优点**：样本独立，训练更像普通监督学习。
- **缺点**：前一个 batch 的隐藏状态不能自然传给下一个 batch。


In [10]:
import random
import torch
from torch import Tensor
from collections.abc import Iterator

def seq_data_iter_random(
    corpus: list[int],
    batch_size: int,
    num_steps: int
) -> Iterator[tuple[Tensor, Tensor]]:
    """使用随机采样生成一个小批量子序列。

    Args:
        corpus: 整个语料库的索引序列。
        batch_size: 每个批次包含的子序列数量。
        num_steps: 每个子序列的时间步长度。

    Yields:
        形状均为 (batch_size, num_steps) 的输入与标签张量。
    """
    # 初始化随机偏移
    start_offset = random.randint(0, num_steps - 1)
    corpus = corpus[start_offset:]
    
    # 计算可生成的自序列数量
    num_subseqs = (len(corpus) - 1) // num_steps

    # 初始化序列 (并打乱)
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    random.shuffle(initial_indices)

    # 数据提取辅助函数
    def data(pos: int) -> list[int]:
        return corpus[pos: pos + num_steps]

    num_batches = num_subseqs // batch_size
    for batch_start in range(0, num_batches * batch_size, batch_size):
        batch_indices = initial_indices[batch_start: batch_start + batch_size]
        x = [data(index) for index in batch_indices]
        y = [data(index + 1) for index in batch_indices]
        yield torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

#### 3.2 顺序分区 (Sequential Partitioning)

保留语料原有顺序，把一整段长序列按 batch 维度切开。

- **优点**：相邻 batch 在时间上连续，适合传递隐藏状态。
- **缺点**：样本相关性更强，打乱程度较小。


In [11]:
def seq_data_iter_sequential(
    corpus: list[int],
    batch_size: int,
    num_steps: int
) -> Iterator[tuple[Tensor, Tensor]]:
    """使用顺序分区生成一个小批量子序列。

    该策略保留相邻批次之间的时间连续性。
    """
    # 这里 num_steps 和 num_steps - 1 应该都行，但是由于 d2l 是 num_steps，所以我也采用了这个。
    offset = random.randint(0, num_steps)
    num_tokens = ((len(corpus) - offset - 1) // batch_size) * batch_size
    xs = torch.tensor(corpus[offset: offset + num_tokens], dtype=torch.long)
    ys = torch.tensor(corpus[offset + 1: offset + 1 + num_tokens], dtype=torch.long)

    xs = xs.reshape(batch_size, -1)
    ys = ys.reshape(batch_size, -1)
    num_batches = xs.shape[1] // num_steps

    for batch_start in range(0, num_batches * num_steps, num_steps):
        x = xs[:, batch_start: batch_start + num_steps]
        y = ys[:, batch_start: batch_start + num_steps]
        yield x, y


---

### 4. 构造序列数据加载器


In [13]:
class SeqDataLoader:
    """时光机器语料的序列数据加载器。"""

    def __init__(
        self,
        batch_size: int,
        num_steps: int,
        use_random_iter: bool,
        max_tokens: int = 10000
    ) -> None:
        self.corpus, self.vocab = load_corpus_time_machine(
            max_tokens=max_tokens,
            token_type='char'
        )
        self.batch_size = batch_size
        self.num_steps = num_steps
        self.use_random_iter = use_random_iter

    def __iter__(self) -> Iterator[tuple[Tensor, Tensor]]:
        data_iter_fn = seq_data_iter_random if self.use_random_iter else seq_data_iter_sequential
        return data_iter_fn(self.corpus, self.batch_size, self.num_steps)

def load_data_time_machine(
    batch_size: int,
    num_steps: int,
    use_random_iter: bool = False,
    max_tokens: int = 10000
) -> tuple[SeqDataLoader, Vocab]:
    """统一返回时光机器序列数据加载器与词表。"""
    data_loader = SeqDataLoader(batch_size, num_steps, use_random_iter, max_tokens=max_tokens)
    return data_loader, data_loader.vocab

# --- 采样策略演示 ---
demo_corpus = list(range(35))
random.seed(40)
print('随机采样示例:')
for x, y in seq_data_iter_random(demo_corpus, batch_size=2, num_steps=5):
    print('X =', x)
    print('Y =', y)
    break

random.seed(42)
print('\n顺序分区示例:')
for x, y in seq_data_iter_sequential(demo_corpus, batch_size=2, num_steps=5):
    print('X =', x)
    print('Y =', y)
    break

time_machine_iter, time_machine_vocab = load_data_time_machine(
    batch_size=2,
    num_steps=8,
    use_random_iter=False,
    max_tokens=64
)

first_x, first_y = next(iter(time_machine_iter))
print('\n真实语料 batch 形状:', first_x.shape, first_y.shape)
print('第一条输入样本(索引):', first_x[0].tolist())
print('第一条输入样本(字符):', ''.join(time_machine_vocab.to_tokens(first_x[0].tolist())))
print('第一条标签样本(字符):', ''.join(time_machine_vocab.to_tokens(first_y[0].tolist())))


随机采样示例:

X = tensor([[13, 14, 15, 16, 17],
        [ 8,  9, 10, 11, 12]])

Y = tensor([[14, 15, 16, 17, 18],
        [ 9, 10, 11, 12, 13]])

顺序分区示例:

X = tensor([[ 5,  6,  7,  8,  9],
        [19, 20, 21, 22, 23]])

...

Dataset already exists and verified: temp/data/plaintext/timemachine.txt

真实语料 batch 形状:
torch.Size([2, 8])
torch.Size([2, 8])

第一条输入样本(索引):
[9, 2, 1, 3, 5, 13, 2, 1]

第一条输入样本(字符): he time

第一条标签样本(字符): e time m


---

### 5. 细致梳理

1. **训练目标本质**：
    - 每个时间步都在做一个多分类任务。
    - 输入是当前位置之前的上下文，输出是“下一个词元是谁”。
2. **为什么 `Y` 是 `X` 右移一位**：
    - 因为语言模型本质就是 next-token prediction。
    - 这和监督学习 (利用 **Label** 进行训练) 没有冲突，只是标签来自原序列本身。
3. **随机采样 vs 顺序分区**：
    - 随机采样更适合把 batch 当作独立样本看待。
    - 顺序分区更适合后续 RNN 传递隐藏状态。
4. **为什么字符级建模常出现在入门教材里**：
    - 词表小，OOV 问题几乎消失。
    - 训练简单，便于专注理解 RNN 机制。
    - 代价是序列更长、局部语义更弱。

---

### 6. 关键工程暗知识 (Engineering Wisdom)

-  **隐藏状态是否可跨 batch 传递，取决于采样策略**：随机采样通常不能直接传，顺序分区才有时间连续性。
-  **`num_steps` 是一个关键超参数**：太小看不到足够上下文，太大会导致显存占用和梯度传播压力上升。
-  **字符级语料适合教学，不一定适合真实任务**：真实 NLP 更常见的是子词（BPE / WordPiece）而不是纯字符。
-  **N-gram 很重要，但不是终点**：它帮助我们理解“条件概率 + 局部上下文”的思想，而 RNN/Transformer 则是在更大上下文里学习这种映射。

---

### 7. 总结 Checklist

-  **语言模型目标**：理解 $P(x_t \mid x_{<t})$ 的数学本质。
-  **N-gram**：明白 unigram / bigram / trigram 的含义与局限。
-  **监督信号构造**：掌握 `X` 与 `Y` 的错位关系。
-  **随机采样与顺序分区**：知道两者为什么都存在，以及各自适合什么训练方式 (是否保留隐藏状态 $H$)。
-  **数据加载器**：为后续 8.4 的 RNN 从零实现准备好统一输入接口。

---


## 8.4 循环神经网络 (RNN)

> 在前面的章节中，我们学习了如何把文本切成时间步（Time steps），现在我们要构建一个真正具有“记忆”的神经网络。
>
> RNN 的核心思想极其优美：**它不仅看着当前的输入，还看着自己上一秒的“记忆”。**

---

### 1. 核心数学原理：隐藏状态 (Hidden State)

在多层感知机（MLP）中，隐藏层的计算是：
$$\mathbf{H} = \phi(\mathbf{X} \mathbf{W}_{xh} + \mathbf{b}_h)$$

而在 RNN 中，我们引入了**上一时刻的隐藏状态 $\mathbf{H}_{t-1}$**。当前时刻 $t$ 的记忆不仅取决于现在的输入 $\mathbf{X}_t$，还取决于过去的记忆 $\mathbf{H}_{t-1}$：
$$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_{xh} + \mathbf{H}_{t-1} \mathbf{W}_{hh} + \mathbf{b}_h)$$

最后，基于当前的记忆计算输出：
$$\mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{hq} + \mathbf{b}_q$$

*   **$\mathbf{W}_{hh}$**：这是 RNN 的灵魂！它是一个方阵，负责决定“过去的记忆有多少能传递到现在”。

---

### 2. 输入预处理 (One-hot 编码)

我们的输入 `X` 是单词的索引（比如 `[2, 5]`），神经网络不能直接对索引做矩阵乘法。我们需要将其转换为**独热编码 (One-hot Encoding)**。


In [2]:
import torch
from torch import Tensor
from torch.nn import functional as F

def get_one_hot_inputs(X: Tensor, vocab_size: int) -> Tensor:
    """将输入的索引张量转换为独热编码，并调整维度以便按时间步迭代。

    Args:
        X: 形状为 (batch_size, num_steps) 的索引张量。
        vocab_size: 词表大小。

    Returns:
        Tensor: 形状为 (num_steps, batch_size, vocab_size) 的浮点张量。
    """
    # 1. X.T 将形状变为 (num_steps, batch_size)
    # 这样我们在外部循环时，每次取出的就是所有样本在同一时刻 t 的输入
    X_transposed = X.T
    
    # 2. F.one_hot 会在最后增加一维 vocab_size
    # 3. 必须转为 float32 才能参与神经网络的权重相乘
    # 现在的形状是 (S, B, V)
    return F.one_hot(X_transposed, vocab_size).type(torch.float32)

# --- 验证 ---
X_test = torch.tensor([[0, 2], [1, 3]]) # batch_size=2, num_steps=2
vocab_size = 4
X_one_hot = get_one_hot_inputs(X_test, vocab_size)
print(f"One-hot 形状: {X_one_hot.shape}") # 预期:[2, 2, 4] -> (时间步, 批量, 词表)

One-hot 形状: torch.Size([2, 2, 4])



---

### 3. 初始化 RNN 模型参数

我们需要手动初始化 3 组参数：输入到隐藏层、隐藏层到隐藏层、隐藏层到输出层。


In [3]:
def get_rnn_params(
    vocab_size: int, 
    num_hiddens: int, 
    device: torch.device
) -> tuple[Tensor, ...]:
    """初始化 RNN 的所有可学习参数。

    Args:
        vocab_size: 词表大小（即输入维度和输出维度）。
        num_hiddens: 隐藏状态的维度（记忆容量）。
        device: 硬件设备。

    Returns:
        包含所有权重和偏置的元组，且均已开启梯度追踪。
    """
    num_inputs = num_outputs = vocab_size

    # 初始化工具函数
    def normal(shape: tuple[int, ...]) -> Tensor:
        return torch.randn(size=shape, device=device) * 0.01

    # 1. 隐藏层参数
    W_xh = normal((num_inputs, num_hiddens))
    W_hh = normal((num_hiddens, num_hiddens)) # 记忆权重方阵
    b_h = torch.zeros(num_hiddens, device=device)

    # 2. 输出层参数
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)

    # 3. 开启梯度追踪
    params =[W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
        
    return tuple(params)


---

### 4. 初始化隐藏状态

RNN 在处理第一个时间步时，没有“过去的记忆”。我们需要给它一个初始化的全 0 状态。


In [ ]:
def init_rnn_state(
    batch_size: int, 
    num_hiddens: int, 
    device: torch.device
) -> tuple[Tensor]:
    """初始化 RNN 的隐藏状态。

    返回元组是为了与后续更复杂的网络（如 LSTM 有两个状态）保持接口一致性。
    """
    return (torch.zeros((batch_size, num_hiddens), device=device), )


---

### 5. RNN 前向传播逻辑

这里是 RNN 的心脏：一个基于时间步的 `for` 循环。


In [ ]:
def rnn_forward(
    inputs: Tensor, 
    state: tuple[Tensor], 
    params: tuple[Tensor, ...]
) -> tuple[Tensor, tuple[Tensor]]:
    """RNN 的前向传播函数。

    Args:
        inputs: 形状为 (num_steps, batch_size, vocab_size) 的输入张量。
        state: 包含隐藏状态 H 的元组。
        params: 包含网络权重的元组。

    Returns:
        Tuple:
            - outputs: 形状为 (num_steps * batch_size, vocab_size) 的拼接输出。
            - state: 最后一个时间步的隐藏状态（用于传递给下一个 Batch）。
    """
    # 解包参数和状态
    W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs: list[Tensor] =[]

    # 沿着时间步遍历
    # X 的形状为 (batch_size, vocab_size)
    for X in inputs:
        # 1. 更新隐藏状态 (当前输入 + 过去记忆)
        # 使用 tanh 激活函数将记忆限制在 [-1, 1] 之间，防止数值爆炸
        # H = torch.tanh(torch.matmul(X, W_xh) + torch.matmul(H, W_hh) + b_h)
        H = torch.tanh(
            torch.cat((X, H), dim=1) @ torch.cat((W_xh, W_hh), dim=0) + b_h
        )
        
        # 2. 计算当前时刻的输出
        Y = torch.matmul(H, W_hq) + b_q
        outputs.append(Y)
        
    # 3. 拼接输出：将列表中的张量沿第 0 维拼接
    # 最终形状变为 (num_steps * batch_size, vocab_size)
    # 这种形状正好可以直接喂给 nn.CrossEntropyLoss
    return torch.cat(outputs, dim=0), (H,)


---

### 6. 细致梳理：

1.  **为什么转置输入 (`X.T`)？**
    *   原始数据是 `(Batch, Time)`。如果直接遍历，你拿到的是“一个句子里的所有词”，这无法进行并行矩阵计算。
    *   转置后变为 `(Time, Batch)`。你遍历拿到的是 **“所有句子在时刻 $t$ 的第一个词”** 。这样就可以利用 GPU 对整个 Batch 同时执行矩阵乘法。
2.  **为什么激活函数用 `tanh` 而不是 `ReLU`？**
    *   在 MLP 中我们推崇 ReLU。但在 RNN 中，由于 $H_t$ 会在时间步上**反复乘以自身权重 $W_{hh}$**（如果是 100 步就是连乘 100 次）。
    *   如果用 ReLU，正数不设上限，连乘 100 次极易导致**数值爆炸**。
    *   `tanh` 将输出死死按在 $[-1, 1]$ 之间，天然具有稳定数值的作用。
3.  **状态传递 (State Passing)**：
    *   `rnn_forward` 返回了更新后的 `state`。在 8.3 节讲的“顺序分区”采样中，我们会把这个 `state` 喂给下一个 Batch，实现跨 Batch 的长程记忆。
4.  **矩阵拼接优化 (The Concatenation Trick)**：
    *   在数学上，$(X, H_{prev}) \cdot \begin{pmatrix} W_{xh} \\ W_{hh} \end{pmatrix} = X \cdot W_{xh} + H_{prev} \cdot W_{hh}$。
    *   代码实现为 `matmul(torch.cat((X, H), dim=1), torch.cat((W_xh, W_hh), dim=0))`。
    *   这种写法减少了 GPU 的任务启动次数，效率更高。

---

### 7. 总结 Checklist

*   **隐状态公式**：牢记 $H_t$ 包含 $X_t$ 和 $H_{t-1}$ 两部分贡献。
*   **形状变换**：明白为什么需要 `(Time, Batch, Vocab)` 这种奇怪的维度。
*   **One-hot**：理解为什么索引不能直接乘以权重。

---
